# **Sesión: Ejemplo de Agentes para Análisis de datos con CrewAI**

**Objetivo:** Utilizar agentes para analizar CSVs y generar reportes.

---

## **Estructura de la clase**

---

### **1. Introducción a los Agentes (30 minutos)**

#### **¿Qué es un agente?**

1. Tiene un **objetivo**
2. Tiene acceso a **herramientas**
3. Puede ejecutar y planear de manera **autónoma**

![Alt Text](assets/agents_explanation.png)

---

#### **Modelo ReAct**
- **Thought:**
    - Inicia secuencia de 'razonamiento'
    - ¿Qué es lo que necesito hacer?

- **Action:**
    - Decide en una acción -> tiene herramientas/_tools_ disponibles

- **Observation:**
    - Ejecuta su acción con las _tools_ disponibles
    - 'Reflexiona' sobre el resultado de su acción

![Alt Text](assets/react_agents.webp)

---

#### **¿Cuándo usar Agentes?**

1. NO todas las tareas requieren Agentes - KISS (Keep It Simple, Stupid)
2. ¿Requiere capacidades de razonamiento o decisiones en tiempo real? -> Agents, maybe
3. ¿Requieres trabajar con datos no estructurados (imágenes, PDFs, etc.)? -> Agents

---

#### **¿Cómo diseñar Agentes?**

1. ¿Qué quieres lograr?
2. ¿A quiénes necesitas para lograrlo?
3. ¿Qué herramientas les vas a dar para que ejecuten su tarea?
4. Sé lo más **modular** posible

![Alt Text](assets/agent_manager.png)

---


#### **NOTA IMPORTANTE:**

- Hay **MUCHOS** frameworks/maneras de crear agentes, hoy usaremos una de las más sencillas, `crewAI`.

---



### **2. Diseñar un Agente para Análisis de Stocks (30 minutos)**

#### **Caso:**

- Necesitas crear un reporte sobre el GDP de 10 países, con gráficas incluidas. No sabes ejecutar código pero tienes un acceso a LLMs que te pueden ayudar.

- Diseña un equipo de agentes para que te ayuden a crear tu reporte. 

#### Discutir: **¿Cómo divides las tareas?, ¿a quiénes necesitas?**

---

### **3. Implementación de un Agente para Análisis de Datos con crewAI (40 minutos)**

In [1]:
import pandas as pd
from crewai import Crew, Task, Agent, Process
from crewai_tools import SerperDevTool, ScrapeWebsiteTool
from dotenv import load_dotenv
import matplotlib.pyplot as plt

load_dotenv()

True

### **Tools**

- `ScrapeWebsiteTool`: Herramienta que te permite 'scrapear' un website, leer el contenido HTML.
- `SerperDevTool`: Herramienta de búsqueda en Google Search.

In [2]:
scraper_tool = ScrapeWebsiteTool()
online_search_tool = SerperDevTool()

### **Agents**

**Analista de Datos**

In [3]:
data_analyst_agent = Agent(
    role="Data Analyst",
    goal="Monitor and analyze market data in real-time "
         "to identify trends and predict market movements.",
    backstory="Specializing in financial markets, this agent "
              "uses statistical modeling and machine learning "
              "to provide crucial insights. With a knack for data, "
              "the Data Analyst Agent is the cornerstone for "
              "informing trading decisions.",
    verbose=True,
    allow_delegation=False,
    tools = [scraper_tool, online_search_tool]
)

**Estratega de trading**

In [4]:
trading_strategy_agent = Agent(
    role="Trading Strategy Developer",
    goal="Develop and test various trading strategies based "
         "on insights from the Data Analyst Agent.",
    backstory="Equipped with a deep understanding of financial "
              "markets and quantitative analysis, this agent "
              "devises and refines trading strategies. It evaluates "
              "the performance of different approaches to determine "
              "the most profitable and risk-averse options.",
    verbose=True,
    allow_delegation=False,
    tools = [scraper_tool, online_search_tool]
)

**Trade Advisor**

In [5]:
execution_agent = Agent(
    role="Trade Advisor",
    goal="Suggest optimal trade execution strategies "
         "based on approved trading strategies.",
    backstory="This agent specializes in analyzing the timing, price, "
              "and logistical details of potential trades. By evaluating "
              "these factors, it provides well-founded suggestions for "
              "when and how trades should be executed to maximize "
              "efficiency and adherence to strategy.",
    verbose=True,
    allow_delegation=False,
    tools = [scraper_tool, online_search_tool]
)

**Administrador de Riesgos**

In [6]:
risk_management_agent = Agent(
    role="Risk Advisor",
    goal="Evaluate and provide insights on the risks "
         "associated with potential trading activities.",
    backstory="Armed with a deep understanding of risk assessment models "
              "and market dynamics, this agent scrutinizes the potential "
              "risks of proposed trades. It offers a detailed analysis of "
              "risk exposure and suggests safeguards to ensure that "
              "trading activities align with the firm’s risk tolerance.",
    verbose=True,
    allow_delegation=False,
    tools = [scraper_tool, online_search_tool]
)

### **Tasks**

- A cada agente le vamos a asignar tareas de acuerdo a su **objetivo**. 

In [7]:
from datetime import date

today = date.today().strftime(format='%Y-%m-%d')

financial_trading_inputs = {
    'stock_selection': 'AAPL',
    'initial_capital': '100000',
    'start_date': '2025-01-01',
    'today': today,
    'risk_tolerance': 'Medium',
    'trading_strategy_preference': 'Day Trading',
    'news_impact_consideration': True
}

In [8]:
data_analysis_task = Task(
    description=(
        "Continuously monitor and analyze market data for "
        "the selected stock ({stock_selection}) "
        "starting from {start_date} to {today}. "
        "Use statistical modeling and machine learning to "
        "identify trends and predict market movements."
    ),
    expected_output=(
        "Insights and alerts about significant market "
        "opportunities or threats for {stock_selection} "
        "from {start_date} to {today}."
    ),
    agent=data_analyst_agent,
)

In [9]:
strategy_development_task = Task(
    description=(
        "Develop and refine trading strategies based on "
        "the insights from the Data Analyst and "
        "user-defined risk tolerance ({risk_tolerance}). "
        "Consider trading preferences ({trading_strategy_preference}) "
        "and initial capital of {initial_capital}."
        "Consider this timeframe for your analysis tarting from {start_date} to {today}."
    ),
    expected_output=(
        "A set of potential trading strategies for {stock_selection} "
        "that align with the user's risk tolerance."
    ),
    agent=trading_strategy_agent,
)


In [10]:
execution_planning_task = Task(
    description=(
        "Analyze approved trading strategies to determine the "
        "best execution methods for {stock_selection}, "
        "considering current market conditions and optimal pricing."
        "Analyze your strategies according to this timeframe: starting from {start_date} to {today}."
    ),
    expected_output=(
        "Detailed execution plans suggesting how and when to "
        "execute trades for {stock_selection}."
    ),
    agent=execution_agent,
)


In [11]:
risk_assessment_task = Task(
    description=(
        "Evaluate the risks associated with the proposed trading "
        "strategies and execution plans for {stock_selection}. "
        "Provide a detailed analysis of potential risks "
        "and suggest mitigation strategies."
        "Evaluate the risks starting from {start_date} to {today}."
    ),
    expected_output=(
        "A comprehensive risk analysis report detailing potential "
        "risks and mitigation recommendations for {stock_selection}."
    ),
    agent=risk_management_agent,
)

### **Crew**

#### NOTA:

Aunque a cada agente le asignamos `tasks`, internamente hemos asignado un `manager` 
que dado nuestro input le dará instrucciones a cada agente: 

`manager_llm`

In [12]:
#from langchain_openai import ChatOpenAI

financial_trading_crew = Crew(
    agents=[data_analyst_agent, 
            trading_strategy_agent, 
            execution_agent, 
            risk_management_agent],
    
    tasks=[data_analysis_task, 
           strategy_development_task, 
           execution_planning_task, 
           risk_assessment_task],

    # el "manager" interno de los agentes
    #manager_llm=ChatOpenAI(model="gpt-4o", 
    #                       temperature=0.7),

    # por ello, se dice que el proceso es jerárquico -> tienen un manager
    #process=Process.hierarchical,
    verbose=True
)

In [13]:
result = financial_trading_crew.kickoff(inputs=financial_trading_inputs)

# Agent: Data Analyst
## Task: Continuously monitor and analyze market data for the selected stock (AAPL) starting from 2025-01-01 to 2025-02-18. Use statistical modeling and machine learning to identify trends and predict market movements.


# Agent: Data Analyst
## Thought: Thought: I need to search for historical market data for AAPL from January 1, 2025, to February 18, 2025, to perform statistical modeling and provide insights. I will start by searching for AAPL historical data for the specified period.
## Using tool: Search the internet
## Tool Input: 
"{\"search_query\": \"AAPL historical stock data January 1 2025 to February 18 2025\"}"
## Tool Output: 
{'searchParameters': {'q': 'AAPL historical stock data January 1 2025 to February 18 2025', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': 'Apple Inc. (AAPL) Stock Historical Prices & Data - Yahoo Finance', 'link': 'https://finance.yahoo.com/quote/AAPL/history/', 'snippet': 'Discover historical prices fo

In [14]:
from IPython.display import Markdown
Markdown(result.raw)

**Comprehensive Risk Analysis Report for AAPL Trading Strategies (01/01/2025 – 02/18/2025)**

**1. Introduction**

This report evaluates the trading strategies proposed for AAPL in the context of a medium risk tolerance and a day trading strategy. We will analyze potential risks and provide mitigation strategies for the period from January 1, 2025, to February 18, 2025.

**2. Trading Strategies Overview**

- **Momentum Trading Strategy**
  - **Risk Factors**: Sudden market reversals, high volatility in initial hours.
  - **Mitigation**: Utilize trailing stop-loss orders to limit losses if trends reverse.

- **Mean Reversion Strategy**
  - **Risk Factors**: Extended deviations from the mean, unexpected trends due to market sentiment.
  - **Mitigation**: Employ stop-loss orders beyond 1% of traded price, regularly reassess moving average calculations.

- **Breakout Strategy**
  - **Risk Factors**: False breakouts, insufficient volume confirmation.
  - **Mitigation**: Ensure breakout confirmation with volumes significantly above 50 million shares and use technical indicators like RSI for validation.

- **Scalping Strategy**
  - **Risk Factors**: Execution risk during high volatility periods, tight profit margins.
  - **Mitigation**: Maintain strict adherence to stop-loss orders set at 0.5% and closely monitor trading windows.

- **Swing Strategy**
  - **Risk Factors**: Overnight holding risks including news events impacting price gaps.
  - **Mitigation**: Select trades with strong technical support, use indicators like RSI and MACD for timing exits.

**3. Market Conditions & Historical Context**

Based on the extracted Yahoo Finance data for AAPL, key market performance metrics are as follows for January-early February 2025:
- Notable volatility with price highs near $247 and lows around $220.
- Significant volume spikes on January 31st and February 3rd indicating strong market movements ideal for momentum and breakout strategies.
  
**4. Risk Analysis and Recommendations**

- **Market Volatility**: High volatility periods pose a significant risk, particularly for momentum and scalping strategies. To mitigate, traders should remain nimble and responsive to real-time market data.
  
- **Economic Events**: Earnings releases, geopolitical developments, and macroeconomic indicators can severely affect trade outcomes. Traders must continuously monitor and potentially adjust strategies ahead of known events.
  
- **Technology Risks**: Algorithmic and automated trading errors pose execution risks. It's imperative to have a robust fail-safe mechanism for algorithmic implementations.

**5. Conclusion**

While the proposed strategies for AAPL leverage historical market data, traders are exposed to various risks, especially due to high market volatility and the potential for unforeseen economic events. By maintaining disciplined risk management, utilizing protective instruments like stop-loss orders, and staying informed of external factors, the risks can be effectively managed in line with a medium risk tolerance. Continuous monitoring and strategic flexibility will be crucial to optimizing trading outcomes for the specified period. 

Overall, aligning these strategies with the firm's risk tolerance policies and keeping abreast of market conditions can contribute constructively to achieving desired financial objectives while mitigating potential losses.

In [15]:
print(f'Usamos estos tokens: {result.token_usage}')

Usamos estos tokens: total_tokens=78103 prompt_tokens=72828 cached_prompt_tokens=12160 completion_tokens=5275 successful_requests=13
